# CompBench Replicate VIEScore

Scores the successful Replicate-generated outputs under `.out/evaluation/compbench/replicate`. The scorer reads the local `selection.jsonl`, per-model/per-task records under `records/`, and each generated image path from those records; it uses the stored source image and edit mask under `tasks/multi_turn_editing/`. Only `multi_object_add` and `multi_object_remove` are included. Results are cached and summarized under `metrics/` without modifying the inference records.

Scoring uses the original CompBench instruction and matching tight source/output crops around the edit mask, with GEdit-Bench VIEScore `task="tie"` and GPT-4o. A full run may make one paid API evaluation per successful output. Configure `VIESCORE_ROOT` and `VIESCORE_KEY_PATH` in the environment; the key file should contain only your OpenAI API key on its first line. Never put the key in a notebook cell or commit the file.

In [4]:
import os
from pathlib import Path

from framework.utils.project import load_project_environment

PROJECT_ROOT = load_project_environment()
REPLICATE_ROOT = Path(
    os.environ.get("MMEDIT_COMPBENCH_REPLICATE_ROOT")
    or PROJECT_ROOT / ".out" / "evaluation" / "compbench" / "replicate"
).expanduser().resolve()
VIESCORE_ROOT = Path(os.environ.get("VIESCORE_ROOT") or ".").expanduser()
VIESCORE_KEY_PATH = Path(os.environ.get("VIESCORE_KEY_PATH") or ".").expanduser()
if not REPLICATE_ROOT.is_dir():
    raise FileNotFoundError(f"Replicate CompBench result directory not found: {REPLICATE_ROOT}")
print("Replicate results:", REPLICATE_ROOT)
print("VIEScore checkout:", VIESCORE_ROOT if str(VIESCORE_ROOT) != "." else "not configured")
print("API key file:", VIESCORE_KEY_PATH if str(VIESCORE_KEY_PATH) != "." else "not configured")

Replicate results: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/evaluation/compbench/replicate
VIEScore checkout: /Users/nghialt3670/Projects/Step1X-Edit/GEdit-Bench
API key file: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.viescore.key


## Review saved outputs

The next cell discovers successful per-model records and verifies that each matches the sample selection manifest and resolves to local source, mask, and output files. Check the model/task counts before invoking the paid scorer.

In [5]:
from collections import Counter
from IPython.display import display

from evaluation.compbench_viescore import collect_replicate_saved_outputs

saved_outputs = collect_replicate_saved_outputs(REPLICATE_ROOT)
output_counts = Counter((row["model"], row["task"]) for row in saved_outputs)
print(f"Found {len(saved_outputs)} succeeded Replicate outputs.")
display(
    [
        {"model": model, "task": task, "count": count}
        for (model, task), count in sorted(output_counts.items())
    ]
)

Found 576 succeeded Replicate outputs.


[{'model': 'bagel', 'task': 'multi_object_add', 'count': 72},
 {'model': 'bagel', 'task': 'multi_object_remove', 'count': 72},
 {'model': 'flux_kontext', 'task': 'multi_object_add', 'count': 72},
 {'model': 'flux_kontext', 'task': 'multi_object_remove', 'count': 72},
 {'model': 'qwen_image_edit', 'task': 'multi_object_add', 'count': 72},
 {'model': 'qwen_image_edit', 'task': 'multi_object_remove', 'count': 72},
 {'model': 'step1x', 'task': 'multi_object_add', 'count': 72},
 {'model': 'step1x', 'task': 'multi_object_remove', 'count': 72}]

## Score saved outputs

Install GEdit-Bench VIEScore dependencies (including `regex` and `requests`) in this notebook's Python environment, clone [Step1X-Edit](https://github.com/stepfun-ai/Step1X-Edit), and set `VIESCORE_ROOT` to its `GEdit-Bench` directory. Run the scoring cell after reviewing the selected outputs. Existing scores are reused when the model, sample, instruction, output path, size, and modification time match.

In [6]:
from evaluation.compbench_viescore import evaluate_replicate_compbench_outputs

viescore_rows, viescore_summary = evaluate_replicate_compbench_outputs(
    REPLICATE_ROOT,
    viescore_root=VIESCORE_ROOT,
    key_path=VIESCORE_KEY_PATH,
)
print(f"Per-sample VIEScore: {REPLICATE_ROOT / 'metrics' / 'modified_viescore.jsonl'}")
print(f"Summary: {REPLICATE_ROOT / 'metrics' / 'modified_viescore_summary.json'}")
display(viescore_summary)

Replicate CompBench VIEScore:  16%|█▌        | 91/576 [05:57<31:42,  3.92s/sample]


ConnectionError: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))